
- find the same categories
- merge them into one defintino 
- get final graph


In [44]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# I/O

In [45]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os
import pickle
import gravis as gv
from collections import defaultdict

In [46]:
cwd = os.getcwd()

parent_dir = Path(cwd).parent

# Add the parent directory to the Python path
sys.path.append(str(parent_dir))

print(sys.path)

['/Users/gl665/.pyenv/versions/3.11.8/lib/python311.zip', '/Users/gl665/.pyenv/versions/3.11.8/lib/python3.11', '/Users/gl665/.pyenv/versions/3.11.8/lib/python3.11/lib-dynload', '', '/Users/gl665/Documents/Phrame-MentalMaps-1/.venv/lib/python3.11/site-packages', '/Users/gl665/Documents/Phrame-MentalMaps-1', '/Users/gl665/Documents/Phrame-MentalMaps-1']


In [47]:

from old_code.cld_nx import cld_nx_properties, combine_graphs_with_node_attributes
from old_code import cld_nx

# Read the data

In [48]:
df = pd.read_csv(
    "/Users/gl665/Documents/Phrame-MentalMaps-1/data/read_for_label/batch_tasks_LLM/results_0318/processed_llm_inferenced_results.csv")

print(df.size)
print(df.head())

591365
   Unnamed: 0                    sentence1  \
0           0  fulfilling ones basic needs   
1           1  fulfilling ones basic needs   
2           2  fulfilling ones basic needs   
3           3  fulfilling ones basic needs   
4           4  fulfilling ones basic needs   

                                           sentence2  similarity  category  
0  the need for instrumental and or emotional sup...      0.4793       0.0  
1  stress resulting from the lack of meeting fina...      0.4188       0.0  
2  having time and or being in an environment in ...      0.5134       0.0  
3              participating in prosocial activities      0.5111       0.0  
4  the extent to which treatment that an individu...      0.4457       0.0  


In [49]:
# swap cell names to vars.
# pickle

with open(
    "/Users/gl665/Documents/Phrame-MentalMaps-1/data/data_dictionary/all_vars_def_dictionary.pkl",
    "rb",
) as f:
    all_vars_def_dictionary = pickle.load(f)

In [50]:
all_vars_def_dictionary

{'alcohol misuse': [['alcohol misuse',
   'using alcohol - in reference to misuse']],
 'intimate partner violence': [['intimate partner violence',
   'perpetration or victimization of violence by a or to a romantic partner through physical, sexual, mental, psychological, financial, social means']],
 'mental health': [['mental health',
   '"Increases" in mental health indicate a decline in negative emotions and mental illness and a rise in positive emotionality']],
 'reportable incident to police': [['reportable incident to police',
   'the occurrence of a (physical mostly) violent incident that may warrant police involvement']],
 'reporting incident of intimate partner violence': [['intimate partner violence',
   'perpetration and or victimization of violence by a or to a romantic partner through physical, sexual, mental, psychological, financial, social means'],
  ['reporting incident of intimate partner violence',
   'reporting an IPV event to the police']],
 'institutional racism ag

In [51]:
# all_vars_def_dictionary

def_vars_dict = {
    v[0][-1]: k for k, v in all_vars_def_dictionary.items()
}

def_vars_dict

{'using alcohol - in reference to misuse': 'alcohol misuse',
 'perpetration or victimization of violence by a or to a romantic partner through physical, sexual, mental, psychological, financial, social means': 'physical injury from intimate partner violence',
 '"Increases" in mental health indicate a decline in negative emotions and mental illness and a rise in positive emotionality': 'mental health',
 'the occurrence of a (physical mostly) violent incident that may warrant police involvement': 'reportable incident to police',
 'perpetration and or victimization of violence by a or to a romantic partner through physical, sexual, mental, psychological, financial, social means': 'people getting charged with intimate partner violence',
 'discriminatory treatment that Indigenous people experience by police - typically due to stereotypes of Indigenous people as criminals and or drug users': 'indigenous people stereotyped by police',
 'people getting involved with the police through either c

In [52]:
# mapping

df['vars1'] = df['sentence1'].map(def_vars_dict, na_action='ignore')
df["vars2"] = df["sentence2"].map(def_vars_dict, na_action="ignore")

df.head()

,Unnamed: 0,sentence1,sentence2,similarity,category,vars1,vars2
0,0,fulfilling ones basic needs,the need for instrumental and or emotional sup...,0.4793,0.0,addressing basic needs,social support seeking
1,1,fulfilling ones basic needs,stress resulting from the lack of meeting fina...,0.4188,0.0,addressing basic needs,financial stress
2,2,fulfilling ones basic needs,having time and or being in an environment in ...,0.5134,0.0,addressing basic needs,opportunity to develop social supports
3,3,fulfilling ones basic needs,participating in prosocial activities,0.5111,0.0,addressing basic needs,engagement in prosocial activities
4,4,fulfilling ones basic needs,the extent to which treatment that an individu...,0.4457,0.0,addressing basic needs,mental health treatment efficacy


In [53]:
print(df["category"].value_counts())

# percentage of each category
print(df["category"].value_counts(normalize=True))

category
0.0    110482
1.0      7791
Name: count, dtype: int64
category
0.0    0.934127
1.0    0.065873
Name: proportion, dtype: float64


# Graph with Disjoint Set (Union-Find) to group the same categories into one group

In [54]:
# grouping if category is the same

class UnionFind:
    def __init__(self):
        self.parent = {}
        self.rank = {}

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # path compression
        return self.parent[x]

    def union(self, x, y):
        root_x = self.find(x)
        root_y = self.find(y)
        if root_x != root_y:
            if self.rank[root_x] > self.rank[root_y]:
                self.parent[root_y] = root_x
            elif self.rank[root_x] < self.rank[root_y]:
                self.parent[root_x] = root_y
            else:
                self.parent[root_y] = root_x
                self.rank[root_x] += 1

    def add(self, x):
        if x not in self.parent:
            self.parent[x] = x
            self.rank[x] = 0


def group_sentences(pairs):
    uf = UnionFind()

    for sent1, sent2, label in pairs:
        if label == 1:
            uf.add(sent1)
            uf.add(sent2)
            uf.union(sent1, sent2)

    groups = defaultdict(list)
    for sentence in uf.parent.keys():
        root = uf.find(sentence)
        groups[root].append(sentence)

    return list(groups.values())

In [55]:
# delete cellts with nan in vars1 or vars2
df = df.dropna(subset=["vars1", "vars2"])

sentence_pairs = df[['vars1', 'vars2', 'category']].to_numpy()

sentence_pairs

array([['addressing basic needs', 'social support seeking', 0.0],
       ['addressing basic needs', 'financial stress', 0.0],
       ['addressing basic needs',
        'opportunity to develop social supports', 0.0],
       ...,
       ['addressing basic needs', 'practical flourishing life skills',
        0.0],
       ['addressing basic needs', 'financial management skills', 0.0],
       ['addressing basic needs', 'ability to acquire a job', 0.0]],
      dtype=object)

In [56]:
clusters = group_sentences(sentence_pairs)

cluster_dict = {}
for i, group in enumerate(clusters):
    cluster_dict[group[0]] = group
print(cluster_dict)

{'reluctance to press charges against violent partner': ['reluctance to press charges against violent partner', 'reluctance to report incident to police', 'victim silence', 'reluctance to call police', 'need for healthy boundaries with unhealthy relationships', 'normative acceptance of intimate partner violence', 'perpetrator deflecting blame', 'people minimizing or underreporting the severity of an intimate partner violence event', 'fear of family divide', 'acceptance of sexual assault', 'family member denial or downplaying of violence', 'perpetrator enablement', 'culture of silence', 'failure to protect other family members', 'family retaliation', 'ignored male victims', 'perception of intimate partner violence as crime vs symptom of trauma', 'personal normalization of violence in relationships', 'enabling perpetration of intimate partner violence', 'family support for treatment', 'community acceptance of intimate partner violence', 'family and friend permissive intimate partner viol

In [57]:
# read in the clds dictionary that includes both the vars name and the graph structure


with open("/Users/gl665/Documents/Phrame-MentalMaps-1/data/data_dictionary/clds_dictionary.pkl", "rb") as f:
    cld_dict = pickle.load(f)

In [58]:
cld_dict[1]

{'name': 'E clean model 572020_edited.json',
 'cld_graph': <networkx.classes.digraph.DiGraph at 0x17ff06710>,
 'node_var_def_dict': {'alcohol misuse': [['alcohol misuse',
    'misusing alcohol']],
  'intimate partner violence': [['intimate partner violence',
    'perpetration and or victimization of violence by a or to a romantic partner through physical, sexual, mental, psychological, financial, social means'],
   ['reporting incident of intimate partner violence',
    'reporting an IPV event to the police']],
  'mental health': [['mental health',
    '"Increases" in mental health indicate a decline in negative emotions/mental illness and a rise in positive emotionality']],
  'reportable incident to police': [['reportable incident to police',
    'the occurrence of a (physical mostly) violent incident that may warrant police involvement']],
  'reporting incident of intimate partner violence': [['intimate partner violence',
    'perpetration and or victimization of violence by a or to 

In [59]:
cld_dict[1]['cld_graph'].nodes()

my_dict = cld_dict[1]['node_var_def_dict']

In [60]:
my_dict['alcohol misuse'][0][1]

'misusing alcohol'

In [61]:
def print_graph(G):

    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G)  # Store layout to keep it consistent
    nx.draw_networkx_nodes(G, pos=pos, node_size=20)
    nx.draw_networkx_labels(G, pos=pos, font_size=6)
    nx.draw_networkx_edges(G, pos=pos)  # Added edge drawing

    plt.show()  # Return the figure object

In [62]:
# for i in cld_dict:
#     nxgraph = cld_dict[i]['cld_graph']
#     var_name_to_def_dict = cld_dict[i]['node_var_def_dict']
#     node_def = {node: var_name_to_def_dict[node][0][1]
#                 for node in var_name_to_def_dict.keys()}

#     # print(node_def)

#     cld_nx_properties(nxgraph)

#     # print(nxgraph.nodes())

#     # print_graph(nxgraph)
#     def_nxgraph = nx.relabel_nodes(nxgraph, node_def, copy=True)

#     # save definition graph
#     cld_dict[i]['def_graph'] = def_nxgraph

#     cld_nx_properties(def_nxgraph)

#     # print_graph(def_nxgraph)
#     print("===================================================")

In [63]:
cld_nx.cld_nx_properties(cld_dict[1]["cld_graph"])
cld_nx.cld_nx_node_properties(cld_dict[1]['cld_graph'])
cld_nx.cld_nx_edge_properties(cld_dict[1]['cld_graph'])

Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGraph'>
Directed: True
Number of nodes: 110
Number of edges: 174
Node data:
-------------

Type: <class 'str'> <class 'dict'>
Var Name: alcohol misuse
Attributes: {}
Degree: 8
Type: <class 'str'> <class 'dict'>
Var Name: intimate partner violence
Attributes: {}
Degree: 14
Type: <class 'str'> <class 'dict'>
Var Name: mental health
Attributes: {}
Degree: 9
Type: <class 'str'> <class 'dict'>
Var Name: reportable incident to police
Attributes: {}
Degree: 3
Type: <class 'str'> <class 'dict'>
Var Name: reporting incident of intimate partner violence
Attributes: {}
Degree: 3
Edge data:
-------------
Type: <class 'str'> <class 'str'> <class 'dict'>
Source: alcohol misuse
Target: intimate partner violence
Attributes: {'weight': 1}
	weight: 1
Type <class 'numpy.int64'>
	weight: 1
Type <class 'numpy.int64'>
Type: <class 'str'> <class 'str'> <class 'dict'>
Source: alcohol misuse
Target: mental health
Attributes: {'weight': -1}
	wei

In [64]:
G = cld_nx.clean_graph_for_gravis(cld_dict[1]['cld_graph'])

cld_nx.gravis_settings(G)

In [65]:
gv.d3(G)

In [66]:
# swap the nodes in the graph if it is in the cluster

for cld_key, val in cld_dict.items():

    # save definition graph
    G = cld_dict[cld_key]['cld_graph']

    mapping_dict = {}
    list_nodes = list(G.nodes())
    for node in list_nodes:
        # creating mapping of the node name to the cluster name
        for key, val in cluster_dict.items():
            if node in val:
                mapping_dict[node] = key

    def_nxgraph_merge = nx.relabel_nodes(G, mapping_dict, copy=True)
    cld_dict[cld_key]["cld_graph_merge"] = def_nxgraph_merge
    # cld_dict[i]["def_graph_merge"] = def_nxgraph_merge

# Merge the CLDS!


In [67]:
cld_dict[1]['node_var_def_dict']

{'alcohol misuse': [['alcohol misuse', 'misusing alcohol']],
 'intimate partner violence': [['intimate partner violence',
   'perpetration and or victimization of violence by a or to a romantic partner through physical, sexual, mental, psychological, financial, social means'],
  ['reporting incident of intimate partner violence',
   'reporting an IPV event to the police']],
 'mental health': [['mental health',
   '"Increases" in mental health indicate a decline in negative emotions/mental illness and a rise in positive emotionality']],
 'reportable incident to police': [['reportable incident to police',
   'the occurrence of a (physical mostly) violent incident that may warrant police involvement']],
 'reporting incident of intimate partner violence': [['intimate partner violence',
   'perpetration and or victimization of violence by a or to a romantic partner through physical, sexual, mental, psychological, financial, social means'],
  ['reporting incident of intimate partner violence

In [68]:
list_of_org_G = [cld_dict[i]['cld_graph'] for i in cld_dict]

In [69]:
# save the list_of_graph

from src.data_management.data_saver import DataSaver
# Initialize DataSaver
saver = DataSaver()

# Save each graph individually
for i, graph in enumerate(list_of_org_G):
    output_path = Path(f"data/processed/org_graph_{i}.pkl")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Save as pickle since graphs contain complex objects
    saver.save_data(graph, output_path)

In [70]:

list_of_G = [cld_dict[i]["cld_graph"] for i in cld_dict.keys()]
print(list_of_G)
combined_graph = combine_graphs_with_node_attributes(list_of_G)

clean_combined_graph = cld_nx.clean_up_combined_graph(combined_graph)

cld_nx_properties(combined_graph)

[<networkx.classes.digraph.DiGraph object at 0x17ff06710>, <networkx.classes.digraph.DiGraph object at 0x30b8f3d90>, <networkx.classes.digraph.DiGraph object at 0x30b9120d0>, <networkx.classes.digraph.DiGraph object at 0x30b930650>, <networkx.classes.digraph.DiGraph object at 0x304014b90>, <networkx.classes.digraph.DiGraph object at 0x30402fb50>, <networkx.classes.digraph.DiGraph object at 0x30403acd0>, <networkx.classes.digraph.DiGraph object at 0x3040543d0>, <networkx.classes.digraph.DiGraph object at 0x30406fc90>, <networkx.classes.digraph.DiGraph object at 0x3040a6d50>, <networkx.classes.digraph.DiGraph object at 0x3040bf2d0>, <networkx.classes.digraph.DiGraph object at 0x3040dc790>, <networkx.classes.digraph.DiGraph object at 0x3040ffad0>, <networkx.classes.digraph.DiGraph object at 0x304127a50>, <networkx.classes.digraph.DiGraph object at 0x304158110>, <networkx.classes.digraph.DiGraph object at 0x304189c50>]
Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGrap

In [71]:
cld_nx.cld_nx_properties(combined_graph)
cld_nx.cld_nx_node_properties(combined_graph)
cld_nx.cld_nx_edge_properties(combined_graph)

Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGraph'>
Directed: True
Number of nodes: 1308
Number of edges: 2772
Node data:
-------------

Type: <class 'str'> <class 'dict'>
Var Name: alcohol misuse
Attributes: {}
Degree: 92
Type: <class 'str'> <class 'dict'>
Var Name: intimate partner violence
Attributes: {}
Degree: 134
Type: <class 'str'> <class 'dict'>
Var Name: mental health
Attributes: {}
Degree: 121
Type: <class 'str'> <class 'dict'>
Var Name: reportable incident to police
Attributes: {}
Degree: 3
Type: <class 'str'> <class 'dict'>
Var Name: reporting incident of intimate partner violence
Attributes: {}
Degree: 3
Edge data:
-------------
Type: <class 'str'> <class 'str'> <class 'dict'>
Source: alcohol misuse
Target: intimate partner violence
Attributes: {'weight': [1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1], 'color': ['blue'], 'merge_weight': 1}
	weight: [1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1]
Type <class 'list'>
	color: ['blue']
Type <class 'list'>
	merge_weight: 1
Ty

{'weight': [-1], 'merge_weight': -1}
	weight: [-1]
Type <class 'list'>
	merge_weight: -1
Type <class 'numpy.int64'>
	weight: [-1]
Type <class 'list'>
	merge_weight: -1
Type <class 'numpy.int64'>
Type: <class 'str'> <class 'str'> <class 'dict'>
Source: perception of intimate partner violence as crime vs symptom of trauma
Target: emphasis on cognitive behavior treatment to treat violent perpetrators
Attributes: {'weight': [1], 'merge_weight': 1}
	weight: [1]
Type <class 'list'>
	merge_weight: 1
Type <class 'numpy.int64'>
	weight: [1]
Type <class 'list'>
	merge_weight: 1
Type <class 'numpy.int64'>
Type: <class 'str'> <class 'str'> <class 'dict'>
Source: maintaining employment
Target: family income
Attributes: {'weight': [1], 'merge_weight': 1}
	weight: [1]
Type <class 'list'>
	merge_weight: 1
Type <class 'numpy.int64'>
	weight: [1]
Type <class 'list'>
	merge_weight: 1
Type <class 'numpy.int64'>
Type: <class 'str'> <class 'str'> <class 'dict'>
Source: healing from trauma in therapy
Target:

In [72]:
def clean_graph_for_gravis(graph):
    # Convert all node and edge attributes to JSON serializable types
    for node, attrs in graph.nodes(data=True):
        for key, value in attrs.items():

            if isinstance(value, np.int64):
                attrs[key] = int(value)
            elif isinstance(value,  np.floating):
                attrs[key] = float(value)

    for u, v, attrs in graph.edges(data=True):
        for key, value in attrs.items():
            # if value is a list of integers, convert to the first int element to str
            if isinstance(value, list):
                attrs[key] = str(value[0])
            if isinstance(value, np.integer):
                attrs[key] = str(value)
            elif isinstance(value,  np.floating):
                attrs[key] = float(value)
    return graph


G = clean_graph_for_gravis(combined_graph)

# # cld_nx.gravis_settings(G)
# gv.d3(G)

In [73]:

gv.d3(G,
      )

In [74]:
# after merge

list_of_G = [cld_dict[i]["cld_graph_merge"] for i in cld_dict.keys()]
print(list_of_G)
combined_graph = combine_graphs_with_node_attributes(list_of_G)
clean_combined_graph = cld_nx.clean_up_combined_graph(combined_graph)


cld_nx_properties(combined_graph)

[<networkx.classes.digraph.DiGraph object at 0x108e17210>, <networkx.classes.digraph.DiGraph object at 0x306de2810>, <networkx.classes.digraph.DiGraph object at 0x17e68c190>, <networkx.classes.digraph.DiGraph object at 0x306d9e250>, <networkx.classes.digraph.DiGraph object at 0x306d650d0>, <networkx.classes.digraph.DiGraph object at 0x306d25d50>, <networkx.classes.digraph.DiGraph object at 0x306d1fd10>, <networkx.classes.digraph.DiGraph object at 0x306d05010>, <networkx.classes.digraph.DiGraph object at 0x306caec50>, <networkx.classes.digraph.DiGraph object at 0x306caf490>, <networkx.classes.digraph.DiGraph object at 0x306c8f310>, <networkx.classes.digraph.DiGraph object at 0x306e5a850>, <networkx.classes.digraph.DiGraph object at 0x306e50610>, <networkx.classes.digraph.DiGraph object at 0x3041fc8d0>, <networkx.classes.digraph.DiGraph object at 0x3042027d0>, <networkx.classes.digraph.DiGraph object at 0x304224090>]
Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGrap

In [75]:

G = clean_graph_for_gravis(combined_graph)
gv.d3(G)

# IGNORE

In [76]:
total_pairs = pd.read_csv(
    "/Users/gl665/Documents/Phrame-MentalMaps-1/data/read_for_label/all_definition_pairs.csv"
)

print(total_pairs.shape)

(2299440, 2)
